In [3]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import minimize
import statsmodels.api as sm

plt.rcParams["figure.dpi"] = 120

def _safelog(x, eps=1e-12):
    return np.log(np.clip(np.asarray(x, float), eps, None))

def qlike(f, y, eps=1e-12):
    f = np.clip(np.asarray(f, float), eps, None)
    y = np.asarray(y, float)
    return np.mean(np.log(f) + y/f)

def mse(yhat, y):
    yhat = np.asarray(yhat, float); y = np.asarray(y, float)
    return np.mean((yhat - y)**2)


In [2]:
pip install rpy2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [rpy2]
Note: you may need to restart the kernel to use updated packages.


In [4]:
rv = pd.read_csv('SPY.csv')
rv = rv[["Date", "Volatility", "Type"]]
rv.rename(columns={"Volatility": "RV_daily"},inplace=True)
rv = rv[rv['Type'] == 'QMLE-Trade']
rv.drop(columns=['Type'], inplace=True)
rv = rv.set_index("Date")
rv.index = pd.to_datetime(rv.index)
rv.index.name = "date"

rv.head()

,RV_daily
date,
1996-01-02,0.140261
1996-01-03,0.082399
1996-01-04,0.211454
1996-01-05,0.022647
1996-01-09,0.228727


In [5]:
# returns (CRSP) → ret  (PERMNO = 84398, date is DD/MM/YYYY)
ret = pd.read_csv("returns (crsp).csv")
ret.columns = ret.columns.str.strip()

# pick columns that really exist in this file
date_col = "date" if "date" in ret.columns else "Date"
ret_col  = "RET" if "RET" in ret.columns else "RETX"  # prefer RET, fallback to RETX

ret = ret[[date_col, "PERMNO", ret_col]].copy()
ret = ret[ret["PERMNO"] == 84398].drop(columns=["PERMNO"])

# parse dates BEFORE setting the index (DD/MM/YYYY)
ret[date_col] = pd.to_datetime(ret[date_col], format="%d/%m/%Y", errors="raise")
ret = ret.sort_values(date_col).set_index(date_col)
ret.index.name = "date"

# make numeric returns (handles 'C','B','' etc.), create r
ret["r"] = pd.to_numeric(ret[ret_col], errors="coerce")
ret.drop(columns=[ret_col], inplace=True)
ret.dropna(subset=["r"], inplace=True)

# cap to last available date you mentioned
ret = ret.loc[:"2024-12-29"]

ret.head()


,r
date,
1996-01-02,0.010673
1996-01-03,0.002766
1996-01-04,-0.009529
1996-01-05,-0.002025
1996-01-08,0.003805


In [8]:
df = rv.join(ret, how="inner").dropna()
df.head()


,RV_daily,r
date,,
1996-01-02,0.140261,0.010673
1996-01-03,0.082399,0.002766
1996-01-04,0.211454,-0.009529
1996-01-05,0.022647,-0.002025
1996-01-09,0.228727,-0.017185
